# Time Series Analysis: Rolling Metrics and Trend Detection

This notebook turns noisy daily revenue into weekly and monthly summaries, rolling averages, cumulative growth, and month-over-month change so the underlying business trend is easier to interpret.

## Setup

The notebook uses the processed timestamp dataset in the workspace. If you have another dataset, replace the file path but keep the same column names or adapt them to `date`, `revenue`, and `orders`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/processed/parsed_timestamps.csv')
df = df.rename(columns={'transaction_date': 'date', 'amount': 'revenue'})
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['orders'] = 1
df = df.dropna(subset=['date', 'revenue']).sort_values('date').reset_index(drop=True)
df.head()

## Task 1: Resample Data by Time Period

Weekly and monthly resampling smooths noise and makes comparison easier. We use different aggregations so we can inspect total revenue, order volume, and average order value by period.

In [ ]:
df_ts = df.set_index('date')

# Weekly aggregation
weekly_revenue = df_ts['revenue'].resample('W').sum()
weekly_count = df_ts['orders'].resample('W').count()
weekly_avg = df_ts['revenue'].resample('W').mean()

# Monthly aggregation
monthly_revenue = df_ts['revenue'].resample('M').sum()
monthly_count = df_ts['orders'].resample('M').count()
monthly_avg = df_ts['revenue'].resample('M').mean()

print('Weekly revenue:')
print(weekly_revenue)
print('
Weekly order count:')
print(weekly_count)
print('
Monthly revenue:')
print(monthly_revenue)
print('
Monthly order count:')
print(monthly_count)

highest_period = monthly_revenue.idxmax()
print(f'
Highest revenue period: {highest_period.date()} with ${monthly_revenue.max():,.0f}')

## Task 2: Rolling Window Average

Rolling averages reveal the sustainable direction of the business by smoothing daily volatility. A 7-day window captures short-term movement, while a 30-day window shows the broader trend.

In [ ]:
df['revenue_ma7'] = df['revenue'].rolling(window=7, min_periods=1).mean()
df['revenue_ma30'] = df['revenue'].rolling(window=30, min_periods=1).mean()

plt.figure(figsize=(12, 6))
plt.plot(df['date'], df['revenue'], label='Raw', alpha=0.3)
plt.plot(df['date'], df['revenue_ma7'], label='7-day MA')
plt.plot(df['date'], df['revenue_ma30'], label='30-day MA')
plt.title('Revenue vs Rolling Averages')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.tight_layout()
plt.savefig('output/rolling_avg.png')
plt.show()

print(df[['date', 'revenue', 'revenue_ma7', 'revenue_ma30']].head(10))

## Task 3: Month-over-Month Percentage Change

Month-over-month change helps determine whether the business is accelerating, declining, or staying flat. Positive values indicate growth, and negative values indicate decline.

In [ ]:
mom_change = monthly_revenue.pct_change() * 100
growth_months = mom_change[mom_change > 0]
decline_months = mom_change[mom_change < 0]

print('Month-over-month change:')
print(mom_change)
print('
Growth months:')
print(growth_months)
print('
Decline months:')
print(decline_months)

## Task 4: Cumulative Sum

Cumulative revenue shows the total revenue accumulated through time. It is useful for seeing whether growth is compounding or flattening out.

In [ ]:
df['cumulative_revenue'] = df['revenue'].cumsum()

plt.figure(figsize=(12, 6))
plt.plot(df['date'], df['cumulative_revenue'], color='navy')
plt.title('Cumulative Revenue Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Revenue')
plt.tight_layout()
plt.savefig('output/cumulative.png')
plt.show()

print(f"Total revenue: ${df['cumulative_revenue'].iloc[-1]:,.0f}")

## Task 5: Trend Pattern and Business Implications

The key question is not whether raw daily revenue fluctuates, but whether the rolling average indicates a sustainable direction. That determines whether the business is accelerating, declining, or stable.

In [ ]:
recent_ma30 = df['revenue_ma30'].iloc[-30:]
trend_direction = 'flat'
trend_magnitude = 0.0
if len(recent_ma30) >= 2:
    if recent_ma30.iloc[-1] > recent_ma30.iloc[0]:
        trend_direction = 'up'
    elif recent_ma30.iloc[-1] < recent_ma30.iloc[0]:
        trend_direction = 'down'
    if recent_ma30.iloc[0] != 0:
        trend_magnitude = ((recent_ma30.iloc[-1] - recent_ma30.iloc[0]) / recent_ma30.iloc[0]) * 100

analysis = f"""
TREND ANALYSIS:

Rolling Average Trend: {trend_direction.upper()}
Change over last 30 days: {trend_magnitude:.1f}%

Month-over-month growth: {mom_change.dropna().iloc[-1]:.1f}%

Business Implications:
- {['Accelerating growth - maintain current strategy', 'Declining momentum - investigate causes', 'Stable trend - continue monitoring'][0 if trend_direction == 'up' else 1 if trend_direction == 'down' else 2]}
- Revenue volatility: ${df['revenue'].std():.0f} (measure of noise)
- Suggested action: {['Keep investing in growth drivers and protect the trajectory', 'Investigate product, pricing, or acquisition issues', 'Optimize efficiency while tracking for a directional break'][0 if trend_direction == 'up' else 1 if trend_direction == 'down' else 2]}
"""

print(analysis)

## Submission

Submit the completed notebook and the GitHub PR link together. If either one is missing, the submission is incomplete.

GitHub PR link: [paste your PR URL here]